# Soluzioni — capitolo 3 (esercizi 1–4 misurati)

In [ ]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import torch, numpy as np
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
logit = torch.randn(64, 10)
print("es. 1:", torch.softmax(logit, dim=1).shape, logit.argmax(dim=1).shape, (logit.argmax(dim=1) == 3).sum().item(), "| argmax(dim=0):", logit.argmax(dim=0).shape)
print("es. 2: righe a media zero:", (logit - logit.mean(dim=1, keepdim=True)).mean(dim=1).abs().max().item() < 1e-6)

In [ ]:
tr = transforms.ToTensor(); mtr = datasets.MNIST("../data", train=True, download=True, transform=tr); mte = datasets.MNIST("../data", train=False, download=True, transform=tr); tdl = DataLoader(mte, batch_size=1000)
def mlp(): return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
def run(m, opt, epoche=3):
    dl = DataLoader(mtr, batch_size=64, shuffle=True); fn = nn.CrossEntropyLoss(); acc = []
    for e in range(epoche):
        m.train()
        for xb, yb in dl: opt.zero_grad(); fn(m(xb), yb).backward(); opt.step()
        m.eval(); c = 0
        with torch.no_grad():
            for xb, yb in tdl: c += (m(xb).argmax(1) == yb).sum().item()
        acc.append(c / 10000)
    return acc
fissa_seme(42); m = mlp(); print("es. 3, SGD lr 0.1 + momentum 0.9:", [f"{a:.1%}" for a in run(m, torch.optim.SGD(m.parameters(), lr=0.1, momentum=0.9))])
fissa_seme(42); m = mlp(); nn.init.zeros_(m[1].weight); nn.init.zeros_(m[1].bias); print("es. 4, solo primo strato a zero:", [f"{a:.1%}" for a in run(m, torch.optim.SGD(m.parameters(), lr=0.1))])